# Comparing the old and new dwarf galaxy VACs

With DESI DR1, a VAC of dwarf galaxies was published. However, the associated paper is still a work-in-progress and
we reached out to the author for additional documentation. The authors have since created a new catalogue with a 
different selection function and methods. This notebook aims to understand the difference between the official (published)
catalogue and the new catalogue recieved through private communication (presumably made public with the paper).

In a dwarf galaxy catalogue, we principally care about the selection function. By reviewing the code, the new catalogue is selected to have:
- Good DESI redshift less than 0.5 or 0.4 and greater than 0.001
- Basic spectroscopic and photometric quality cuts
- Photometric quality (reprocessing photometry in dubious cases)
- Mass

In [ ]:
import matplotlib.pyplot as plt
from matplotlib_venn import venn2

import numpy as np
from astropy.table import Table, hstack, join

In [ ]:
import arya
arya.use("mnras")

In [ ]:
import seaborn as sns

In [ ]:
from astropy.cosmology import Planck18, z_at_value

In [ ]:
from astropy import units as u

In [ ]:
import sys
sys.path.append("..")

from desi_utils.plotting import density_scatter

In [ ]:
plt.rcParams["savefig.directory"] = "./figures"

In [ ]:
plt.rcParams["savefig.directory"]

# Load data

In [ ]:
dwarfs_old = Table.read("../data/stellar_mass_emline_dwarfs.fits")

In [ ]:
dwarf_main = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=1)
dwarf_fsf = Table.read("../data/desi_dr1_dwarf_catalog.fits", hdu=4)

In [ ]:
dwarfs_new = hstack([dwarf_main, dwarf_fsf])

# Comparison of Distributions

In [ ]:
def overlap_stats(x, y, xlabel="x", ylabel="y", plot=True, ax=None, **kwargs):
    x = np.unique(x)
    y = np.unique(y)
    Nx = len(x)
    Ny = len(y)
    Nxy = np.sum(np.isin(x, y))  # elements of x found in y
    Nyx = np.sum(np.isin(y, x))  # elements of y found in x

    print(f"number {xlabel}: {Nx}")
    print(f"number {ylabel}: {Ny}")
    print(f"{Nxy} / {Nx} = {Nxy / Nx:0.4f} of {xlabel} are in {ylabel}")
    print(f"{Nyx} / {Ny} = {Nyx / Ny:0.4f} of {ylabel} are in {xlabel}")

    if plot:
        if ax is None:
            fig, ax = plt.subplots()

        only_x = Nx - Nxy
        only_y = Ny - Nyx
        v = venn2(subsets=(only_x, only_y, Nxy), set_labels=(xlabel, ylabel), ax=ax)

        both_count = f"{Nxy}" if Nxy == Nyx else f"{Nxy} / {Nyx}"
        texts = {
            "10": f"{only_x:,}",
            "01": f"{only_y:,}",
            "11": f"{Nxy:,}",
        }
        for region_id, text in texts.items():
            label = v.get_label_by_id(region_id)
            if label is not None:  # region is None when its count is 0
                label.set_text(text)
                label.set_fontsize(9)



    return Nx, Ny, Nxy, Nyx

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))

overlap_stats(dwarfs_old["targetid"], dwarfs_new["TARGETID_1"], xlabel="old", ylabel="new", ax=ax)
plt.title("Overlap of old and new dwarf galaxy catalogue")
plt.savefig("figures/old_new_dwarf_vac_venn.pdf")

The above figure shows that the new catalogue is smaller than the old (470k vs 645k) and the overlap is reasonable byt about 1/3 of the new catalogue were not in the old catalogue. The selection function must have significantly changed between releases.

In [ ]:
def to_finite_numpy(x):
    x_out = np.asarray(x)
    return x_out[np.isfinite(x_out)]

In [ ]:
def compare_hist(x_old, x_new, ax=None, bins=None):
    if ax is None:
        ax = plt.gca()
        
    x_old = to_finite_numpy(x_old)
    x_new = to_finite_numpy(x_new)
    ax.hist(x_old, bins=bins, histtype="step", label="old", density=True)
    ax.hist(x_new, bins=bins, histtype="step", label="new", density=True)

    ax.set(
        ylabel = "density"
    )
    ax.legend()

In [ ]:
plt.rcParams["text.usetex"] = False

In [ ]:
fig, ax = plt.subplots()

compare_hist(dwarfs_old["z"], dwarfs_new["Z"])

ax.set(
    xlabel = "redshift",
    xlim=(0, 0.5)
)

plt.savefig("figures/z_old_new_hist.pdf")

In [ ]:
fig, ax = plt.subplots()

compare_hist(dwarfs_old["mag_r"], dwarfs_new["MAG_R"])

ax.set(
    xlabel = "R magnitude"
)
plt.savefig("figures/rmag_old_new_hist.pdf")

In [ ]:
fig, ax = plt.subplots()

compare_hist(dwarfs_old["logm_cigale"][dwarfs_old["logm_cigale"] > 0], dwarfs_new["LOG_MSTAR_M24"])

ax.set(
    xlabel = "log stellar mass"
)
plt.savefig("figures/mass_old_new_hist.pdf")

In [ ]:
fig, ax = plt.subplots()

compare_hist(dwarfs_old["mag_g"] - dwarfs_old["mag_r"], dwarfs_new["MAG_G"] - dwarfs_new["MAG_R"])

ax.set(
    xlabel = "g - r mag",
    yscale="log",
    ylim=1e-2,
)
plt.savefig("figures/g_r_old_new_hist.pdf")

In [ ]:
fig, ax = plt.subplots()

compare_hist(np.log10(dwarfs_old["deltachi2"]), np.log10(dwarfs_new["DELTACHI2"]))

ax.set(
    xlabel = "DESI $\\Delta \\xi^2$ template fit (higher is better)",
    yscale="log", 
)
plt.savefig("figures/delta_chi2_hist.pdf")

# Quality flags

In [ ]:
np.mean(dwarfs_new["DWARF_MASKBIT"] == 0)

# 2D distributions

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))

plt.sca(axs[0])
density_scatter(dwarfs_old["ra"], dwarfs_old["z"], ylim=(0, 0.5), xlim=(0, 360))
plt.xlabel("RA")
plt.ylabel("z")
plt.title("old")

plt.sca(axs[1])
density_scatter(dwarfs_new["RA_TARGET"], dwarfs_new["Z"], ylim=(0, 0.51), xlim=(0, 360))
plt.xlabel("RA")
plt.ylabel("z")
plt.title("new")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))

plt.sca(axs[0])
density_scatter(dwarfs_old["ra"], dwarfs_old["z"], ylim=(0, 0.1), xlim=(0, 360))
plt.xlabel("RA")
plt.ylabel("z")
plt.title("old")

plt.sca(axs[1])
density_scatter(dwarfs_new["RA_TARGET"], dwarfs_new["Z"], ylim=(0, 0.1), xlim=(0, 360))
plt.xlabel("RA")
plt.ylabel("z")
plt.title("new")

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(6, 3))

plt.sca(axs[0])
density_scatter(dwarfs_old["z"], dwarfs_old["logm_cigale"], xlim=(0, 0.5), ylim=(4.5, 9.6))
plt.xlabel("z")
plt.ylabel("log M cigale")
plt.title("old")


plt.sca(axs[1])
density_scatter(dwarfs_new["Z"], dwarfs_new["LOG_MSTAR_M24"], xlim=(0, 0.5), ylim=(4.5, 9.6))
plt.xlabel("z")
plt.ylabel("log M M24")
plt.title("new")

In [ ]:
log_M_stars = np.arange(6.5, 9.5, 0.5)

fig, axs = plt.subplots(2, 3, figsize=(6, 3), sharex=True, sharey=True)

for i in range(len(log_M_stars)):
    ii = i // 3
    jj = i % 3
    plt.sca(axs[ii, jj])

    logM_min = log_M_stars[i]
    logM = dwarfs_old["logm_cigale"]
    filt = logM > logM_min
    filt &= logM_min + 0.5 > logM

    density_scatter(dwarfs_old["z"][filt], (dwarfs_old["mag_g"] - dwarfs_old["mag_r"])[filt], xlim=(0, 0.5), ylim=(-1, 1), scatter_below=3)
    # plt.annotate(f"$\\log M_\\star \in [{logM_min}, {logM_min + 0.5}]$", (0.5, -1.0), (-0.5, -0.5), va="bottom", ha="right", textcoords="offset fontsize")
    plt.title(f"$\\log M_\\star \in [{logM_min}, {logM_min + 0.5}]$")
    if ii == 1:
        plt.xlabel("redshift")

    if jj == 0:
        plt.ylabel("g - r colour")

plt.tight_layout()

plt.savefig("figures/gr_redshift_mass_old.pdf")

In [ ]:
log_M_stars = np.arange(6.5, 9.5, 0.5)

fig, axs = plt.subplots(2, 3, figsize=(6, 3), sharex=True, sharey=True)

for i in range(len(log_M_stars)):
    ii = i // 3
    jj = i % 3
    plt.sca(axs[ii, jj])

    logM_min = log_M_stars[i]
    logM = dwarfs_new["LOG_MSTAR_M24"]
    filt = logM > logM_min
    filt &= logM_min + 0.5 > logM

    density_scatter(dwarfs_new["Z"][filt], (dwarfs_new["MAG_G"] - dwarfs_new["MAG_R"])[filt], xlim=(0, 0.5), ylim=(-1, 1), scatter_below=3)
    # plt.annotate(f"$\\log M_\\star \in [{logM_min}, {logM_min + 0.5}]$", (0.5, -1.0), (-0.5, -0.5), va="bottom", ha="right", textcoords="offset fontsize")
    plt.title(f"$\\log M_\\star \in [{logM_min}, {logM_min + 0.5}]$")
    if ii == 1:
        plt.xlabel("redshift")

    if jj == 0:
        plt.ylabel("g - r colour")

plt.tight_layout()

plt.savefig("figures/gr_redshift_mass_new.pdf")

# Crossmatched quantities

In [ ]:
dwarfs_new["targetid"] = dwarfs_new["TARGETID_1"]

In [ ]:
dwarfs_matched = join(dwarfs_old, dwarfs_new, "targetid", table_names=("old", "new"))

In [ ]:
fig, ax = plt.subplots()
density_scatter(dwarfs_matched["z"], dwarfs_matched["Z"], scatter_below=10);
plt.xlabel("redshift (old)")
plt.ylabel("redshift (new)")

ax.set_aspect(1)

In [ ]:
fig, ax = plt.subplots()
density_scatter(dwarfs_matched["mag_r"], dwarfs_matched["MAG_R"]);
plt.xlabel("r mag (old)")
plt.ylabel("r mag (new)")

ax.set(aspect=1)

In [ ]:
fig, ax = plt.subplots()
density_scatter(dwarfs_matched["logm_saga"], dwarfs_matched["LOG_MSTAR_M24"],
               xlim=(5, 9.25), ylim =(5, 9.25));
plt.xlabel("log M* phot old")
plt.ylabel("log M* phot new")

ax.set(aspect=1)

In [ ]:
fig, ax = plt.subplots()

xlim = (6, 9.5)
density_scatter(np.log10(dwarfs_matched["mass_cg"]), dwarfs_matched["LOG_MSTAR_M24"], 
                xlim=xlim, ylim=xlim)

plt.xlabel("cigale log M$_\\star$")
plt.ylabel("de los Reyes log M$_\\star$")

plt.plot(xlim, xlim)

ax.set(aspect=1)

In [ ]:
d_L = Planck18.luminosity_distance(dwarfs_matched["Z"])

In [ ]:
M_r = dwarfs_matched["MAG_R"] - 5*np.log10(d_L.to("pc").value / 10)

In [ ]:
plt.hist(M_r);

In [ ]:
log_L_star = -0.4 * (M_r - 4.83)

In [ ]:
density_scatter(dwarfs_matched["MAG_G"] - dwarfs_matched["MAG_R"], log_L_star - dwarfs_matched["LOG_MSTAR_M24"]);

In [ ]:
M_L = np.log10(dwarfs_matched["LOG_MSTAR_M24"] - dwarfs_matched["M

In [ ]:
fig, ax = plt.subplots()

density_scatter(np.log10(dwarfs_matched["mass_cg"]), dwarfs_matched["LOG_MSTAR_M24"] - np.log10(dwarfs_matched["mass_cg"]), xlim=(5, 10), ylim=(-2, 2))

plt.xlabel("cigale log M$_\\star$")
plt.ylabel("de los Reyes - cigale log M$_\\star$")

plt.axhline(0)
ax.set(aspect=1)

In [ ]:
fig, ax = plt.subplots()

density_scatter(np.log10(dwarfs_matched["flux_scale"]), np.log10(dwarfs_matched["APERCORR"]))

ax.set(
    xlabel = "flux scale (old)",
    ylabel = "apeture corr. (new)",
    aspect = 1
)



# The excluded galaxies

In [ ]:
dwarfs_old_lost = dwarfs_old[~np.isin(dwarfs_old["targetid"], dwarfs_new["TARGETID_1"])]

In [ ]:
dwarfs_new_added = dwarfs_new[~np.isin(dwarfs_new["TARGETID_1"], dwarfs_old["targetid"])]

In [ ]:
bins = np.linspace(0, 0.4, 100)
plt.hist(dwarfs_old["z"], bins=bins)
plt.hist(dwarfs_old_lost["z"], bins=bins);

In [ ]:

plt.hist(dwarfs_old["mag_r"])
plt.hist(dwarfs_old_lost["mag_r"]);

In [ ]:
bins = np.linspace(-1, 1.5, 100)
plt.hist(dwarfs_old["mag_g"] - dwarfs_old["mag_r"], bins=bins)
plt.hist(dwarfs_old_lost["mag_g"] - dwarfs_old_lost["mag_r"], bins=bins);
plt.yscale("log")
plt.ylim(1)

In [ ]:
plt.hist(dwarfs_old["fracflux_z"], bins=np.linspace(0, 1, 100))
plt.hist(dwarfs_old_lost["fracflux_z"], bins=np.linspace(0, 1, 100))

plt.yscale('log')

In [ ]:
plt.hist(